In [17]:
 # ── Core libraries ──────────────────────────────────────────────
import requests
import json
import re
import time
from datetime import datetime
 
# ── Data handling ────────────────────────────────────────────────
import pandas as pd
 
# ── NLP ──────────────────────────────────────────────────────────
import nltk
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
 
# ── Web Scraping ─────────────────────────────────────────────────
from bs4 import BeautifulSoup
 
# ── Visualization ─────────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from wordcloud import WordCloud
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')
 
print("✅ All imports successful!")

✅ All imports successful!


In [1]:
NEWS_API_KEY = "de6d5996e1424cb9a6e7015868f7f902"
CATEGORIES = ["technology", "business"]
SEARCH_QUERIES = ["economy","artificial intelligence"]
PAGE_SIZE = 2
COUNTRY = "us"
BASE_URL = "https://newsapi.org/v2"



In [24]:
all_articles = []

In [27]:
def fetch_top_headlines(category:str, country:str=COUNTRY , page_size:int=PAGE_SIZE) -> list:
    url = f"{BASE_URL}/top-headlines"
    params = {
        "apiKey": NEWS_API_KEY,
        "category": category,
        "country": country,
        "pageSize": page_size
    }
    print (params)
    
    response = requests.get(url, params=params)    
    print (response)
    if response.status_code != 200:
        print(f"API ERROR: {response.status_code} - {response.text}")
    articles = response.json().get("articles", [])
    print(articles)
    return articles
    
for cat in CATEGORIES:
    articles = fetch_top_headlines(category=cat,country=COUNTRY)
    for art in articles:
        art["query_category"] = cat
    all_articles.extend(articles)

print(all_articles)

        


{'apiKey': 'de6d5996e1424cb9a6e7015868f7f902', 'category': 'technology', 'country': 'us', 'pageSize': 2}
<Response [200]>
[{'source': {'id': 'the-verge', 'name': 'The Verge'}, 'author': 'Jay Peters', 'title': 'Anthropic essentially bans OpenClaw from Claude by making subscribers pay extra - The Verge', 'description': 'The popular combination of OpenClaw and Claude Code is being severed now that Anthropic has announced it will start charging subscribers extra to access its AI with third-party tools.', 'url': 'https://www.theverge.com/ai-artificial-intelligence/907074/anthropic-openclaw-claude-subscription-ban', 'urlToImage': 'https://platform.theverge.com/wp-content/uploads/sites/2/2026/02/STKB382_OPEN_CLAW_D.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200', 'publishedAt': '2026-04-03T23:52:49Z', 'content': '<ul><li></li><li></li><li></li></ul>\r\nClaude subscriptions will no longer cover third-party access from tools like OpenClaw starting Saturday, Apr

In [28]:
def fetch_by_keyword(query:str, country:str=COUNTRY , page_size:int=PAGE_SIZE) -> list:
    url = f"{BASE_URL}/everything"
    request = {
        "apiKey": NEWS_API_KEY,
        "q": query,
        "language": "en",
        "sortBy": "publishedAt",
        "pageSize": page_size
    }
    print (request)
    
    response = requests.get(url, params=request)    
    print (response)
    if response.status_code != 200:
        print(f"API ERROR: {response.status_code} - {response.text}")
    articles = response.json().get("articles", [])
    print(articles)
    return (articles)

for query in SEARCH_QUERIES:
    articles = fetch_by_keyword(query=query) 
    for art in articles:
        art["query_category"] = query 
    all_articles.extend(articles)
print(all_articles)


{'apiKey': 'de6d5996e1424cb9a6e7015868f7f902', 'q': 'economy', 'language': 'en', 'sortBy': 'publishedAt', 'pageSize': 2}
<Response [200]>
[{'source': {'id': None, 'name': 'Socialobserver.in'}, 'author': 'Social Observer', 'title': 'Gold slips 17% from peak: Should buyers step in now amid global signals? - Social Observer', 'description': 'Gold dips 17% from peak to ~₹1.49L/10g; global price ~$4,679. US-Iran tension lifts oil, revives inflation fears, caps gains. Buy or wait?', 'url': 'https://socialobserver.in/business/gold-slips-17-from-peak-should-buyers-step-in-now-amid-global-signals/', 'urlToImage': 'https://socialobserver.in/wp-content/uploads/2026/04/gold.jpg', 'publishedAt': '2026-04-04T06:17:42Z', 'content': 'New Delhi – Gold prices corrected sharply from record highs. Then, fresh global cues reshaped investor sentiment. Now, buyers face a key questiondoes this dip offer a real opportunity?\r\nGold in India… [+2796 chars]'}, {'source': {'id': 'the-times-of-india', 'name': 'The

In [29]:
for art in all_articles:
    print(art)
    

{'source': {'id': 'the-verge', 'name': 'The Verge'}, 'author': 'Jay Peters', 'title': 'Anthropic essentially bans OpenClaw from Claude by making subscribers pay extra - The Verge', 'description': 'The popular combination of OpenClaw and Claude Code is being severed now that Anthropic has announced it will start charging subscribers extra to access its AI with third-party tools.', 'url': 'https://www.theverge.com/ai-artificial-intelligence/907074/anthropic-openclaw-claude-subscription-ban', 'urlToImage': 'https://platform.theverge.com/wp-content/uploads/sites/2/2026/02/STKB382_OPEN_CLAW_D.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200', 'publishedAt': '2026-04-03T23:52:49Z', 'content': '<ul><li></li><li></li><li></li></ul>\r\nClaude subscriptions will no longer cover third-party access from tools like OpenClaw starting Saturday, April 4th.\r\nClaude subscriptions will no longer cover th… [+3077 chars]', 'query_category': 'technology'}
{'source': {'id': 

In [31]:
print(json.dumps(all_articles, indent=2))

[
  {
    "source": {
      "id": "the-verge",
      "name": "The Verge"
    },
    "author": "Jay Peters",
    "title": "Anthropic essentially bans OpenClaw from Claude by making subscribers pay extra - The Verge",
    "description": "The popular combination of OpenClaw and Claude Code is being severed now that Anthropic has announced it will start charging subscribers extra to access its AI with third-party tools.",
    "url": "https://www.theverge.com/ai-artificial-intelligence/907074/anthropic-openclaw-claude-subscription-ban",
    "urlToImage": "https://platform.theverge.com/wp-content/uploads/sites/2/2026/02/STKB382_OPEN_CLAW_D.jpg?quality=90&strip=all&crop=0%2C10.732984293194%2C100%2C78.534031413613&w=1200",
    "publishedAt": "2026-04-03T23:52:49Z",
    "content": "<ul><li></li><li></li><li></li></ul>\r\nClaude subscriptions will no longer cover third-party access from tools like OpenClaw starting Saturday, April 4th.\r\nClaude subscriptions will no longer cover th\u2026 [+3077

In [34]:
def articles_to_dataframe(articles:list) -> pd.DataFrame:
    rows = []
    for art in articles:
        rows.append({
            "source": art.get("source", {}).get("name"),
            "author": art.get("author"),
            "title": art.get("title"),
            "description": art.get("description"),
            "url": art.get("url"),
            "publishedAt": art.get("publishedAt"),
            "content": art.get("content"),
            "query_category": art.get("query_category"),
            "full_text": f"{art.get('title', '')} {art.get('description', '')} {art.get('content', '')}"    
        })
    df = pd.DataFrame(rows) 
    return df
df = articles_to_dataframe(all_articles)
df.head()

,source,author,title,description,url,publishedAt,content,query_category,full_text
0,The Verge,Jay Peters,Anthropic essentially bans OpenClaw from Claud...,The popular combination of OpenClaw and Claude...,https://www.theverge.com/ai-artificial-intelli...,2026-04-03T23:52:49Z,<ul><li></li><li></li><li></li></ul>\r\nClaude...,technology,Anthropic essentially bans OpenClaw from Claud...
1,VentureBeat,Carl Franzen,Karpathy shares 'LLM Knowledge Base' architect...,Karpathy proposes something simpler and more l...,https://venturebeat.com/data/karpathy-shares-l...,2026-04-03T23:31:02Z,AI vibe coders have yet another reason to than...,technology,Karpathy shares 'LLM Knowledge Base' architect...
2,CBS News,Aimee Picchi,3.1 million bottles of eye drops sold at Walgr...,The eye drops — sold under multiple brands — h...,https://www.cbsnews.com/news/eye-drops-recall-...,2026-04-04T03:57:13Z,More than 3.1 million bottles of eye drops sol...,business,3.1 million bottles of eye drops sold at Walgr...
3,The Verge,Jay Peters,Anthropic essentially bans OpenClaw from Claud...,The popular combination of OpenClaw and Claude...,https://www.theverge.com/ai-artificial-intelli...,2026-04-03T23:52:49Z,<ul><li></li><li></li><li></li></ul>\r\nClaude...,business,Anthropic essentially bans OpenClaw from Claud...
4,Socialobserver.in,Social Observer,Gold slips 17% from peak: Should buyers step i...,Gold dips 17% from peak to ~₹1.49L/10g; global...,https://socialobserver.in/business/gold-slips-...,2026-04-04T06:17:42Z,New Delhi – Gold prices corrected sharply from...,economy,Gold slips 17% from peak: Should buyers step i...
